In [1]:
import torch
from transformers import CLIPTokenizer
import pandas as pd
import textwrap
from datetime import datetime

from made.data_pipeline.utils import set_plotting_configuration
set_plotting_configuration()

import matplotlib.pyplot as plt

from made.data_pipeline.data.datacomp_handler import decode_webdataset, get_next_batch
from made.data_pipeline.utils import collect_tar_files
from made.paths import MADE_PATH
from made.models.meru.nn import model_init, image_to_numpy



In [2]:
model, trs = model_init(pretrained=str(MADE_PATH / "models/ckpt.pt"))
model = model.to("cuda").eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
references = torch.load(str(MADE_PATH / "models/reference.pt"))
img_ref = references["img"].to("cuda")
txt_ref = references["txt"].to("cuda")

In [ ]:
dataset = iter(decode_webdataset(
    collect_tar_files("/home/leobaro/Downloads/datasets/web/datacomp/full_datacomp_multimodal_alignment_filter_output"),
    get_images=True,
    get_captions=True,
    batch_size=50
))
i_dataset = iter(dataset)
curvature = model.curvature.exp()

In [4]:
uid_score = {}

In [ ]:
from made.data_pipeline.filtering_functions.multimodal_filters import _entailment
import torch

def _image_specificity(txt_ref: torch.Tensor, curv: float, image: torch.Tensor):
    txt_ref = txt_ref.to(image.device)
    ient = _entailment(txt_ref, image, curv)
    return ient.mean(dim=0)

def _text_specificity(img_ref: torch.Tensor, curv: float, text: torch.Tensor):    
    img_ref = img_ref.to(text.device)
    tent = _entailment(text, img_ref, curv)
    return tent.mean(dim=1)


In [ ]:
c = 0
while True:
    batch = get_next_batch(i_dataset)
    if batch is None:
        print("Batch is None")
        break
    # if c > 0 and c % 1000 == 0:
    #     print(f"Processed {c} samples. Breaking..")
    #     break
    uids, images, captions = batch
    try:
        images = torch.stack([trs(im) for im in images]).to("cuda")

        with torch.no_grad():
            encoded_images = model.encode_image(images)

        with torch.no_grad():
            tokenized_captions = tokenizer(captions, return_tensors="pt", padding="max_length", truncation=True, max_length=77)["input_ids"].to("cuda")
            encoded_captions = model.encode_text(tokenized_captions)

        image_specificity_scores = _image_specificity(txt_ref=encoded_captions, curv=curvature, image=encoded_images)
        text_specificity_scores = _text_specificity(img_ref=encoded_images, curv=curvature, text=encoded_captions)

        for uid, image, img_ss, caption, txt_ss in zip(uids, images, image_specificity_scores, captions, text_specificity_scores):
            uid_score[uid] = (img_ss.item(), txt_ss.item(), uid, image_to_numpy(image), caption)
        c += len(uids)
        print(f"Processed {c} samples")
    except Exception as e:
        print(e)
        continue


In [ ]:
len(uid_score)

In [25]:
for uid, tuple_elements in uid_score.items():
    uid_score[uid] = (tuple_elements[0], tuple_elements[1], tuple_elements[2], tuple_elements[3], tuple_elements[4])

In [26]:
image_specificity_scores = {uid: tuple_elements[0] for uid, tuple_elements in uid_score.items()}
text_specificity_scores = {uid: tuple_elements[1] for uid, tuple_elements in uid_score.items()}

In [ ]:
pd.DataFrame(image_specificity_scores.values()).describe()

In [ ]:
pd.DataFrame(text_specificity_scores.values()).describe()

In [ ]:
from made.data_pipeline.utils import set_plotting_configuration
set_plotting_configuration()
import matplotlib.pyplot as plt
import numpy as np
w=0.7
merged_specificity_scores = [iss*(1-w)+tss*w for iss,tss in zip(image_specificity_scores, text_specificity_scores)]
plt.hist(image_specificity_scores, bins=50, label="Image specificity", alpha=0.5)
plt.hist(text_specificity_scores, bins=100, label="Text specificity", alpha=0.5)
plt.hist(merged_specificity_scores, bins=100, label="Specificity", alpha=0.5)

plt.title("Specificity scores distribution (DFN)")
plt.xlabel("Specificity score")
plt.ylabel("Frequency")
#plt.axvline(x=np.mean(sim_scores), color='red', linestyle='--', label='mean')
#plt.axvline(x=np.percentile(sim_scores, 25), color='grey', linestyle='--', label='25th and 75th percentiles')
#plt.axvline(x=np.percentile(sim_scores, 75), color='grey', linestyle='--')
plt.legend()
plt.savefig("specificity_scores_distribution.png")
plt.show()


In [ ]:
percentile_ranges = [(x,x+10) for x in range(0, 100, 10)]
percentile_ranges

In [31]:
image_specificity_scores = [tuple_elements[0] for uid, tuple_elements in uid_score.items()]
images = [tuple_elements[3] for uid, tuple_elements in uid_score.items()]

In [32]:
sorted_indices = sorted(range(len(image_specificity_scores)), key=lambda i: image_specificity_scores[i])
sorted_image_specificity_scores = [image_specificity_scores[i] for i in sorted_indices]
sorted_images = [images[i] for i in sorted_indices]

In [ ]:
percentile_ranges_min_max = []
for percentile_range in percentile_ranges:
    p_min, p_max = np.percentile(sorted_image_specificity_scores, percentile_range[0]), np.percentile(sorted_image_specificity_scores, percentile_range[1])
    print(f"{percentile_range[0]}-{percentile_range[1]}", round(p_min, 3), round(p_max, 3))
    percentile_ranges_min_max.append((p_min, p_max))

In [ ]:
percentile_ranges_min_max

In [35]:
def get_samples_and_scores_from_percentile_range(p_min, p_max, scores, samples):
    ok_indexes = [i for i, score in enumerate(scores) if p_min <= score <= p_max]
    return [samples[i] for i in ok_indexes], [scores[i] for i in ok_indexes]

In [ ]:
num_rows = len(percentile_ranges_min_max)
num_cols = 6

plt.rcParams['figure.autolayout'] = False
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows*2.2))

for j, percentile_range in enumerate(percentile_ranges_min_max):
    
    samples, scores = get_samples_and_scores_from_percentile_range(
        percentile_range[0], percentile_range[1], sorted_image_specificity_scores, sorted_images)
    

    for i, (image, score) in enumerate(zip(samples[0:num_cols], scores[0:num_cols])):
        axes[j][i].imshow(image)
        #axes[j][i].set_title(f"{round(score, 4)}")
        axes[j][i].axis("off")

#plt.subplots_adjust(hspace=0.3, wspace=0.2)        
plt.tight_layout(pad=0.1)
fig.savefig("images_specificity_percentiles.png", dpi=150)
plt.show()

In [37]:
text_specificity_scores = [tuple_elements[1] for uid, tuple_elements in uid_score.items()]
captions = [tuple_elements[4] for uid, tuple_elements in uid_score.items()]

In [38]:
sorted_indices = sorted(range(len(text_specificity_scores)), key=lambda i: text_specificity_scores[i])
sorted_text_specificity_scores = [text_specificity_scores[i] for i in sorted_indices]
sorted_captions = [captions[i] for i in sorted_indices]

In [ ]:
text_percentile_ranges_min_max = []
for percentile_range in percentile_ranges:
    p_min, p_max = np.percentile(sorted_text_specificity_scores, percentile_range[0]), np.percentile(sorted_text_specificity_scores, percentile_range[1])
    print(f"{percentile_range[0]}-{percentile_range[1]}", round(p_min, 3), round(p_max, 3))
    text_percentile_ranges_min_max.append((p_min, p_max))

In [40]:
with open("text_percentile_ranges_min_max.txt", "w") as f:

    for j, text_percentile_range in enumerate(text_percentile_ranges_min_max):
        samples, scores = get_samples_and_scores_from_percentile_range(
            text_percentile_range[0], text_percentile_range[1], sorted_text_specificity_scores, sorted_captions)
        f.write(f"Text percentile range {j}: {round(text_percentile_range[0], 3)}-{round(text_percentile_range[1], 3)}\n")
        for sample, score in zip(samples[0:10], scores[0:10]):
            f.write(f"\t{sample} {round(score, 3)}\n")
        f.write("\n")
